# Superstore — analisi di profittabilità

**Come usare questo notebook:** esegui le celle una alla volta (`Shift+Invio`), dall'alto verso il basso. Prima di ogni cella di codice trovi *cosa fa* e *cosa cercare nell'output*. Le conclusioni le scrivi tu nell'ultima sezione.

**Le analisi richieste:**
1. Profitto per categoria
2. Profitto per sottocategoria
3. Profitto per stato
4. Profitto per segmento
5. Impatto degli sconti sul profitto
6. I 10 prodotti che generano le maggiori perdite

## Step 1 — Caricare e ispezionare i dati

Prima regola: mai analizzare dati che non hai ispezionato. Questa cella carica il CSV e mostra dimensioni, duplicati, nulli e le statistiche delle colonne numeriche.

Nota: il file è codificato `windows-1252` (formato tipico dei CSV esportati da Excel su Windows). Senza quel parametro `read_csv` va in errore.

**Cosa cercare nell'output:**
- Quante righe? Ogni riga è un intero ordine o un prodotto dentro un ordine? (guarda se `Order ID` si ripete)
- Ci sono nulli o duplicati da pulire?
- Guarda `min` e `max` di `Profit`: cosa noti?
- `Discount` è espresso come percentuale (20) o frazione (0.20)?

In [ ]:
import pandas as pd

df = pd.read_csv("Sample - Superstore.csv", encoding="windows-1252")

print("righe x colonne:", df.shape)
print("duplicati:", df.duplicated().sum())
print("nulli totali:", int(df.isna().sum().sum()))
print("Order ID unici:", df["Order ID"].nunique(), "(su", len(df), "righe)")

df[["Sales", "Quantity", "Discount", "Profit"]].describe().round(2)

In [ ]:
# Le prime righe, per prendere confidenza con le colonne
df.head(3)

## Step 2 — Definire le metriche

Per ogni dimensione (categoria, stato, ...) calcoleremo sempre due numeri:

- **Profitto totale** = `sum(Profit)` → dice *dove* si guadagna o si perde
- **Margine** = `sum(Profit) / sum(Sales)` → dice *quanto bene* si vende

Attenzione a un errore comune: il margine è il **rapporto delle somme**, non la media dei margini riga per riga. `mean(Profit/Sales)` darebbe lo stesso peso a una vendita da 2$ e a una da 20.000$.

Definiamo una funzione riutilizzabile, così le quattro analisi diventano quattro chiamate.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def profilo(df, dim):
    """Profitto totale e margine per una dimensione, ordinati per profitto."""
    return (df.groupby(dim)
              .agg(Sales=("Sales", "sum"),          # fatturato
                   Profit=("Profit", "sum"),        # profitto totale
                   Righe=("Row ID", "count"))       # numero righe d'ordine
              .assign(Margine_pct=lambda t: (t["Profit"] / t["Sales"] * 100).round(1))
              .sort_values("Profit", ascending=False))

# Il riferimento: i totali dell'azienda
tot_s, tot_p = df["Sales"].sum(), df["Profit"].sum()
print(f"Totale: vendite ${tot_s:,.0f} | profitto ${tot_p:,.0f} | margine {tot_p/tot_s:.1%}")

## Step 3 — Analisi 1: profitto per categoria

**Cosa cercare:** confronta le colonne `Sales` e `Profit` insieme. C'è una categoria che fattura tanto ma rende poco? Il margine dell'azienda è 12,5%: chi sta sopra, chi sta sotto?

In [ ]:
profilo(df, "Category")

## Step 4 — Analisi 2: profitto per sottocategoria

Quando una categoria mostra un'anomalia, la mossa standard è **scendere di un livello** (drill-down) per capire chi la causa.

**Cosa cercare:** ci sono sottocategorie con profitto negativo? A quale categoria appartengono? Confronta con quello che hai visto allo Step 3.

In [ ]:
profilo(df, "Sub-Category")

## Step 5 — Analisi 3: profitto per stato

49 stati sono troppi per leggerli tutti: si guardano i migliori, i peggiori, e quanti sono in perdita.

**Cosa cercare:** gli stati in perdita sono stati che vendono poco? Oppure ce n'è qualcuno con fatturato alto e profitto negativo? (quello è il caso interessante: il volume non è il suo problema)

In [ ]:
stati = profilo(df, "State")

print("stati in perdita:", (stati["Profit"] < 0).sum(), "su", len(stati))
print("\n--- top 5 ---")
display(stati.head(5))
print("--- bottom 5 ---")
display(stati.tail(5))

## Step 6 — Analisi 4: profitto per segmento

**Cosa cercare:** i margini dei tre segmenti sono molto diversi tra loro? Se sono simili, anche questo è un risultato: significa che la profittabilità non dipende dal tipo di cliente — e restringe il campo delle spiegazioni possibili.

In [ ]:
profilo(df, "Segment")

## Step 7 — Formulare un'ipotesi

Fermati e rileggi i quattro output. Questo è il momento in cui il data scientist smette di calcolare e ragiona:

- Quali sottocategorie perdono? Che tipo di prodotti sono?
- Lo stato che perde di più vende poco o vende tanto?
- Allo Step 1 hai visto la mediana di `Discount`: quante righe sono scontate?

Se gli indizi puntano nella stessa direzione, formuli un'**ipotesi verificabile**. Per esempio: *"le perdite si concentrano dove gli sconti sono alti"*. Il passo dopo è testarla sui dati — mai fermarsi all'intuizione.

## Step 8 — Analisi 5: impatto degli sconti sul profitto

`Discount` assume pochi valori distinti (0, 0.10, 0.20, ...): possiamo raggruppare per livello esatto e vedere il profitto a ogni livello. Questo è più informativo di una correlazione, perché mostra la **forma** della relazione.

**Cosa cercare:** a quale livello di sconto il profitto totale cambia segno? Il passaggio è graduale o brusco?

In [ ]:
(df.groupby("Discount")
   .agg(Righe=("Row ID", "count"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
   .assign(Margine_pct=lambda t: (t["Profit"] / t["Sales"] * 100).round(1)))

Ora quantifichiamo il rischio per fascia: che percentuale di righe è in perdita a ogni fascia di sconto?

**Cosa cercare:** la quota di righe in perdita cresce piano o fa un salto? Dove?

In [ ]:
fasce = pd.cut(df["Discount"], bins=[-0.01, 0, 0.2, 0.3, 0.8],
               labels=["0%", "1-20%", "21-30%", ">30%"])

df.groupby(fasce, observed=True).agg(
    Righe=("Row ID", "count"),
    Quota_in_perdita=("Profit", lambda s: f"{(s < 0).mean():.0%}"),
    Profit_tot=("Profit", "sum"))

**Controprova.** Se l'ipotesi "le perdite seguono gli sconti" è vera, i casi anomali trovati prima (lo stato che perde vendendo tanto, la sottocategoria peggiore) devono avere sconti sopra la media. Verifichiamo — sostituisci pure stato e sottocategoria con quelli che hai individuato tu.

In [ ]:
stato_sospetto = "Texas"        # <-- lo stato che hai individuato allo Step 5
subcat_sospetta = "Tables"      # <-- la sottocategoria dello Step 4

print(f"sconto medio {stato_sospetto}: {df.loc[df.State == stato_sospetto, 'Discount'].mean():.0%}"
      f"  |  resto US: {df.loc[df.State != stato_sospetto, 'Discount'].mean():.0%}")
print(f"sconto medio {subcat_sospetta}: {df.loc[df['Sub-Category'] == subcat_sospetta, 'Discount'].mean():.0%}"
      f"  |  resto: {df.loc[df['Sub-Category'] != subcat_sospetta, 'Discount'].mean():.0%}")

# La correlazione lineare, per confronto: nota quanto è debole
# rispetto a quello che hai appena visto nelle tabelle. Perché?
print("correlazione Discount-Profit:", round(df["Discount"].corr(df["Profit"]), 3))

## Step 9 — Analisi 6: i 10 prodotti che perdono di più

Qui si aggrega per **prodotto** (uno stesso prodotto compare su più righe d'ordine) e ci si porta dietro le colonne che possono spiegare la perdita: sottocategoria e sconto medio.

**Cosa cercare:** da quali sottocategorie vengono questi prodotti? Che sconti medi hanno? Quanti ordini bastano per generare quelle perdite? Torna tutto con l'ipotesi dello Step 7?

In [ ]:
pd.set_option("display.max_colwidth", 50)

top_loss = (df.groupby("Product Name")
              .agg(SubCat=("Sub-Category", "first"),
                   Ordini=("Order ID", "nunique"),
                   Sales=("Sales", "sum"),
                   Sconto_medio=("Discount", "mean"),
                   Profit=("Profit", "sum"))
              .sort_values("Profit")     # ascendente: le perdite peggiori in cima
              .head(10))

print(f"perdita cumulata top 10: ${top_loss['Profit'].sum():,.0f}")
top_loss

## Step 10 — I grafici

Principi usati qui sotto:

- **La forma segue il dato.** Valori tutti positivi → barre di un solo colore. Valori positivi *e* negativi → barre divergenti (blu = profitto, rosso = perdita), con la linea dello zero in evidenza.
- **Etichette dirette** sui valori: chi legge non deve interpolare sulla griglia.
- **Un grafico = un messaggio**, dichiarato nel titolo. "Profitto per sottocategoria" è una didascalia; il titolo dice cosa mostra il grafico.
- Mai doppi assi y; griglia leggera dietro le barre.

La coppia blu/rosso è verificata per i daltonici. La prossima cella imposta lo stile una volta per tutte.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

Path("grafici").mkdir(exist_ok=True)   # i PNG finiscono qui

BLU, ROSSO = "#2a78d6", "#e34948"
GRID, INK2, MUTED, BASE = "#e1e0d9", "#52514e", "#898781", "#c3c2b7"

mpl.rcParams.update({
    "font.family": "Segoe UI",
    "axes.edgecolor": BASE, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110,
})

def usd(v):
    """Formato compatto: 55600 -> $55.6k"""
    return f"${v/1000:,.1f}k" if abs(v) >= 1000 else f"${v:,.0f}"

def stile_hbar(ax):
    """Rifiniture per barre orizzontali: griglia verticale leggera, niente trattini."""
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)              # griglia DIETRO le barre
    ax.tick_params(length=0)
    ax.xaxis.set_major_formatter(lambda v, _: usd(v))

def salva(fig, nome):
    fig.tight_layout()
    fig.savefig(f"grafici/{nome}.png", dpi=150, bbox_inches="tight")

### Grafico 1 — Profitto per categoria

Tre valori, tutti positivi → barre orizzontali di un colore. Il margine va annotato accanto al valore: è l'informazione che il profitto da solo non dà.

Dopo averlo generato, prova a cambiare il titolo: qual è il messaggio di questo grafico secondo te?

In [ ]:
cat = profilo(df, "Category").sort_values("Profit")   # ascendente: la barra più lunga in alto

fig, ax = plt.subplots(figsize=(8.5, 2.8))
bars = ax.barh(cat.index, cat["Profit"], color=BLU, height=0.58)

# etichetta diretta: valore + margine
for b, p, m in zip(bars, cat["Profit"], cat["Margine_pct"]):
    ax.text(b.get_width() + 3500, b.get_y() + b.get_height()/2,
            f"{usd(p)}  ·  margine {m:.1f}%", va="center", color=INK2, fontsize=10)

stile_hbar(ax)
ax.set_xlim(0, 205000)                 # spazio a destra per le etichette
ax.set_title("Profitto per categoria", loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "01_categoria")
plt.show()

### Grafico 2 — Profitto per sottocategoria

Qui ci sono valori negativi: il colore codifica il **segno** (blu profitto, rosso perdita) e la linea dello zero fa da spartiacque. Le barre sono ordinate per valore, così le perdite si raccolgono in basso.

In [ ]:
sub = profilo(df, "Sub-Category").sort_values("Profit")

fig, ax = plt.subplots(figsize=(8.5, 6.2))
bars = ax.barh(sub.index, sub["Profit"], height=0.65,
               color=[ROSSO if p < 0 else BLU for p in sub["Profit"]])
ax.bar_label(bars, fmt=usd, padding=4, fontsize=8.5, color=INK2)
ax.axvline(0, color=BASE, linewidth=1)          # la linea dello zero
stile_hbar(ax)
ax.set_xlim(-26000, 66000)
ax.set_title("Profitto per sottocategoria", loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "02_sottocategoria")
plt.show()

### Grafico 3 — Profitto per stato

49 barre sarebbero illeggibili: si mostrano i 10 migliori e i 10 peggiori. Va detto nel titolo o in didascalia, altrimenti il grafico mente per omissione.

In [ ]:
sel = pd.concat([stati.head(10), stati.tail(10)]).sort_values("Profit")

fig, ax = plt.subplots(figsize=(8.5, 6.8))
bars = ax.barh(sel.index, sel["Profit"], height=0.65,
               color=[ROSSO if p < 0 else BLU for p in sel["Profit"]])
ax.bar_label(bars, fmt=usd, padding=4, fontsize=8.5, color=INK2)
ax.axvline(0, color=BASE, linewidth=1)
stile_hbar(ax)
ax.set_xlim(-33000, 90000)
ax.set_title("Profitto per stato — primi e ultimi 10",
             loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "03_stati")
plt.show()

### Grafico 4 — Profitto per segmento

Stessa forma del grafico 1. Il messaggio qui è la *somiglianza* dei margini, quindi l'annotazione del margine è la parte importante.

In [ ]:
seg = profilo(df, "Segment").sort_values("Profit")

fig, ax = plt.subplots(figsize=(8.5, 2.8))
bars = ax.barh(seg.index, seg["Profit"], color=BLU, height=0.58)
for b, p, m in zip(bars, seg["Profit"], seg["Margine_pct"]):
    ax.text(b.get_width() + 3000, b.get_y() + b.get_height()/2,
            f"{usd(p)}  ·  margine {m:.1f}%", va="center", color=INK2, fontsize=10)
stile_hbar(ax)
ax.set_xlim(0, 185000)
ax.set_title("Profitto per segmento", loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "04_segmento")
plt.show()

### Grafico 5 — Profitto per livello di sconto

L'asse x è ordinato (0% → 80%), quindi barre verticali. Il colore segna di nuovo il segno: il punto in cui le barre passano da blu a rosso È il finding.

In [ ]:
sc = df.groupby("Discount")["Profit"].sum()

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.bar([f"{d:.0%}" for d in sc.index], sc.values, width=0.66,
              color=[ROSSO if v < 0 else BLU for v in sc.values])
ax.bar_label(bars, fmt=usd, padding=3, fontsize=8.5, color=INK2)
ax.axhline(0, color=BASE, linewidth=1)

# stile per barre verticali: griglia orizzontale
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.yaxis.set_major_formatter(lambda v, _: usd(v))

ax.set_xlabel("Sconto applicato")
ax.set_title("Profitto totale per livello di sconto",
             loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "05_sconti")
plt.show()

### Grafico 6 — I 10 prodotti con le maggiori perdite

Tutte perdite → tutte rosse. L'etichetta aggiunge lo sconto medio: collega visivamente questo grafico all'analisi degli sconti.

In [ ]:
top10 = top_loss.iloc[::-1]        # invertito: il peggiore finisce in alto
nomi = [n if len(n) <= 42 else n[:39] + "..." for n in top10.index]

fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(nomi, top10["Profit"], color=ROSSO, height=0.65)
for b, p, s in zip(bars, top10["Profit"], top10["Sconto_medio"]):
    ax.text(b.get_width() - 150, b.get_y() + b.get_height()/2,
            f"{usd(p)} · sconto medio {s:.0%}", va="center", ha="right",
            color=INK2, fontsize=8.5)
ax.axvline(0, color=BASE, linewidth=1)
stile_hbar(ax)
ax.set_xlim(-13500, 0)
ax.set_title("I 10 prodotti con le maggiori perdite",
             loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "06_top10_perdite")
plt.show()

---
# Analisi aggiuntive

Le domande a cui rispondere sono quattro:
1. Quali prodotti generano perdite? *(→ Step 4 e 9)*
2. Quali aree geografiche hanno le performance peggiori? *(→ Step 5, e ora Step 11)*
3. Gli sconti riducono la redditività? *(→ Step 8, e ora Step 13 e 14)*
4. Ci sono stati con vendite elevate ma profitto basso? *(→ Step 12, l'analisi fatta apposta)*

Gli step che seguono usano lo stile grafico definito allo Step 10: esegui prima quella cella se hai riavviato il kernel.

## Step 11 — Profitto per Region

Prima del dettaglio per stato conviene il quadro a 4 macro-aree: costa quasi zero (la funzione `profilo` è già pronta) e dà la prima risposta alla domanda 2.

**Cosa cercare:** quale regione ha il margine più basso? Confronta con gli stati in perdita dello Step 5: si concentrano in quella regione?

In [ ]:
profilo(df, "Region")

## Step 12 — Vendite vs profitto per stato (scatter)

La domanda 4 incrocia **due misure**: vendite e profitto. Quando la domanda è un incrocio, la forma giusta è lo scatter: ogni stato è un punto, e la domanda diventa "chi sta nel quadrante vendite alte / profitto negativo?".

Etichettiamo solo i punti notevoli (vendite molto alte o perdite pesanti): etichettare tutti i 49 stati renderebbe il grafico illeggibile.

**Cosa cercare:** quali stati stanno sotto la linea dello zero *e* a destra? Quelli sono la risposta alla domanda 4 — vendere di più non li salverebbe.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))

ax.scatter(stati["Sales"], stati["Profit"], s=45, alpha=0.85,
           c=[ROSSO if p < 0 else BLU for p in stati["Profit"]],
           edgecolor="white", linewidth=0.8)
ax.axhline(0, color=BASE, linewidth=1)          # sopra = guadagno, sotto = perdita

# etichette solo per i punti notevoli
notevoli = stati[(stati["Sales"] > 150_000) | (stati["Profit"] < -10_000)]
for nome, r in notevoli.iterrows():
    ax.annotate(nome, (r["Sales"], r["Profit"]),
                xytext=(6, 4), textcoords="offset points", fontsize=8.5, color=INK2)

ax.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.xaxis.set_major_formatter(lambda v, _: usd(v))
ax.yaxis.set_major_formatter(lambda v, _: usd(v))
ax.set_xlabel("Vendite")
ax.set_ylabel("Profitto")
ax.set_title("Vendite vs profitto per stato", loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "07_stati_scatter")
plt.show()

## Step 13 — Sottocategorie in perdita: colpa del prodotto o dello sconto?

Allo Step 4 hai trovato sottocategorie in perdita. Due spiegazioni possibili:
- **il prodotto** costa troppo o è prezzato male → perderebbe anche a prezzo pieno;
- **lo sconto** → a prezzo pieno guadagna, scontato perde.

Il test: spaccare ogni sottocategoria in perdita tra righe a prezzo pieno e righe scontate, e confrontare i margini.

**Cosa cercare:** a prezzo pieno il margine è positivo? Se sì, il prodotto è sano e il problema è la politica commerciale. Se è negativo anche a prezzo pieno, il problema è il prodotto stesso.

In [ ]:
sub_in_perdita = ["Tables", "Bookcases", "Supplies"]   # <-- quelle trovate allo Step 4

tmp = df[df["Sub-Category"].isin(sub_in_perdita)].copy()
tmp["Prezzo"] = tmp["Discount"].gt(0).map({False: "pieno", True: "scontato"})

(tmp.groupby(["Sub-Category", "Prezzo"])
    .agg(Righe=("Row ID", "count"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .assign(Margine_pct=lambda t: (t["Profit"] / t["Sales"] * 100).round(1)))

## Step 14 — Trend annuale: problema strutturale o politica recente?

La domanda di business: *il danno da sconti c'è da sempre o è iniziato con una nuova politica commerciale?* Per rispondere servono, anno per anno:
- margine → la redditività sta peggiorando?
- sconto medio e quota di righe scontate oltre il 20% → la politica sconti è cambiata?

**Cosa cercare:** se sconto medio e margine sono piatti sui 4 anni, il problema è **strutturale** (la politica è dannosa da sempre). Se lo sconto sale e il margine scende nell'ultimo anno, è una **deriva recente**. Le due diagnosi portano ad azioni diverse.

In [ ]:
df["Anno"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y").dt.year

trend = (df.groupby("Anno")
           .agg(Sales=("Sales", "sum"),
                Profit=("Profit", "sum"),
                Sconto_medio=("Discount", "mean"),
                Quota_sconto_oltre_20=("Discount", lambda s: (s > 0.2).mean()))
           .assign(Margine_pct=lambda t: (t["Profit"] / t["Sales"] * 100).round(1)))
trend

Il grafico: margine e sconto medio sono entrambi percentuali di scala simile, quindi possono stare sulle **stesse assi** (mai due scale y diverse su un grafico — se le unità non coincidono, si fanno due pannelli). Etichette dirette a fine linea al posto della legenda.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))

ax.plot(trend.index, trend["Margine_pct"], color=BLU, marker="o", linewidth=2)
ax.plot(trend.index, trend["Sconto_medio"] * 100, color=ROSSO, marker="o", linewidth=2)

# etichette dirette a fine linea
ax.text(trend.index[-1] + 0.08, trend["Margine_pct"].iloc[-1],
        "margine %", color=BLU, va="center", fontsize=10, fontweight="bold")
ax.text(trend.index[-1] + 0.08, trend["Sconto_medio"].iloc[-1] * 100,
        "sconto medio %", color=ROSSO, va="center", fontsize=10, fontweight="bold")

ax.set_ylim(0, 25)
ax.set_xticks(trend.index)
ax.set_xlim(trend.index[0] - 0.2, trend.index[-1] + 1.1)   # spazio per le etichette
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
ax.set_title("Margine e sconto medio per anno", loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "08_trend_annuale")
plt.show()

## Step 15 — Dove perdono gli stati in perdita? (heatmap stato × categoria)

Localizza il problema all'incrocio di due dimensioni: per ciascuno dei 10 stati in perdita, il profitto per categoria. Colore divergente (rosso = perdita, blu = profitto) centrato sullo zero, valori scritti nelle celle.

**Cosa cercare:** gli stati in perdita perdono anche su categorie che a livello nazionale sono profittevoli (Step 3)? Se sì, il problema non è il mix di prodotti — è locale. Ricorda lo sconto medio del Texas trovato allo Step 8.

In [ ]:
import matplotlib.colors as mcolors

in_perdita = stati[stati["Profit"] < 0].index
piv = (df[df["State"].isin(in_perdita)]
         .pivot_table(index="State", columns="Category", values="Profit", aggfunc="sum"))
piv = piv.loc[piv.sum(axis=1).sort_values().index]      # il peggiore in alto

lim = float(abs(piv.values).max())                      # scala simmetrica attorno a 0
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.imshow(piv.values, cmap="RdBu", aspect="auto",
          norm=mcolors.TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim))

ax.set_xticks(range(piv.shape[1]), piv.columns)
ax.set_yticks(range(piv.shape[0]), piv.index)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)

# il valore scritto in ogni cella; bianco sulle celle piu' scure
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.iloc[i, j]
        ax.text(j, i, usd(v), ha="center", va="center", fontsize=9,
                color="white" if abs(v) > lim * 0.55 else "#0b0b0b")

ax.set_title("Profitto per categoria negli stati in perdita",
             loc="left", fontsize=12, fontweight="bold", pad=12)
salva(fig, "09_heatmap_stati")
plt.show()

## Step 16 — La mappa: profitto e sconto fianco a fianco

Il test dello Step 8 ha stabilito la relazione sconto→perdita. Le mappe la rendono *visibile*: due choropleth affiancate — profitto per stato e sconto medio per stato — mostrano che sono in gran parte la stessa mappa.

Scelte di colore, seguendo la logica del dato:
- **profitto** = polarità → scala divergente centrata sullo zero (rosso perdita, blu profitto). La scala è tarata a ±30k$ sulle perdite: California e New York (+76k) saturano al massimo, accettabile perché sono outlier positivi;
- **sconto** = magnitudine → scala sequenziale a **una sola tinta** (viola, chiaro→scuro). Non rosso: il rosso qui significa già "perdita", riusarlo per lo sconto confonderebbe i due messaggi.

Serve plotly: se l'import fallisce, esegui prima in una cella `%pip install plotly`.

**Cosa cercare:** gli stati scuri nella mappa di destra sono gli stessi colorati di rosso a sinistra?

In [ ]:
import plotly.express as px

# plotly localizza gli stati USA con la sigla a due lettere
ABBR = {
 "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA","Colorado":"CO",
 "Connecticut":"CT","Delaware":"DE","District of Columbia":"DC","Florida":"FL","Georgia":"GA",
 "Hawaii":"HI","Idaho":"ID","Illinois":"IL","Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY",
 "Louisiana":"LA","Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI","Minnesota":"MN",
 "Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV","New Hampshire":"NH",
 "New Jersey":"NJ","New Mexico":"NM","New York":"NY","North Carolina":"NC","North Dakota":"ND",
 "Ohio":"OH","Oklahoma":"OK","Oregon":"OR","Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC",
 "South Dakota":"SD","Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA",
 "Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY"}

geo = (df.groupby("State")
         .agg(Profit=("Profit", "sum"), Sconto_medio=("Discount", "mean"))
         .reset_index())
geo["code"] = geo["State"].map(ABBR)

# Mappa 1: profitto — scala divergente centrata sullo zero, tarata sulle perdite
fig = px.choropleth(geo, locations="code", locationmode="USA-states", scope="usa",
                    color="Profit", color_continuous_scale="RdBu",
                    range_color=(-30_000, 30_000), color_continuous_midpoint=0,
                    hover_name="State", title="Profitto totale per stato")
fig.show()

In [ ]:
# Mappa 2: sconto medio — scala sequenziale, una sola tinta
fig = px.choropleth(geo, locations="code", locationmode="USA-states", scope="usa",
                    color="Sconto_medio", color_continuous_scale="Purples",
                    hover_name="State", title="Sconto medio per stato")
fig.show()

## Step 17 — Le tue conclusioni

Compila tu, sulla base di quello che hai visto. Schema suggerito per ogni finding: *cosa* (il fatto), *quanto* (il numero), *perché* (il collegamento con le altre analisi).

| # | Finding | Numero chiave |
|---|---------|---------------|
| 1 | ... | ... |
| 2 | ... | ... |
| 3 | ... | ... |
| 4 | ... | ... |
| 5 | ... | ... |
| 6 | ... | ... |

**Raccomandazioni:** se fossi il management, cosa cambieresti domani mattina?

**Limiti:** questi dati sono osservazionali, non un esperimento. Cosa NON puoi concludere con certezza sulla relazione sconti → perdite?

**Risposte alle quattro domande** (una frase + un numero ciascuna):
1. Quali prodotti generano perdite? → ...
2. Quali aree geografiche hanno le performance peggiori? → ...
3. Gli sconti riducono la redditività? → ...
4. Ci sono stati con vendite elevate ma profitto basso? → ...